## Create Table to summarize results

This notebook creates a table with the normalized NLLHs for different models. It expects to find pickled predictions for 10 folds with the following filenames:

 * ```<scenario_name>.floc.<dist_name>.<fold>.100.<model>.pkl```

`eval_model.py` automatically creates such files. 

**NOTE:** The first cell contains some variables that need to be adjusted. Evaluating all scenarios might take a while.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import tabulate

#% matplotlib inline

sys.path.append("../")
from helper import load_data, preprocess, data_source_release

mean_exp = []  #["rf", "nn"]
sc_dict = data_source_release.get_sc_dict()

# 1) Select the distribution you want to fit
floc_exp = ["lognormal_xgb"]
distribution_hndl = scst.distributions.lognorm
# floc_exp = ["expon_xgb"]
# distribution_hndl = scst.distributions.expon
# floc_exp = ["invgauss_xgb"]
# distribution_hndl = scst.distributions.invgauss
# floc_exp = ["weibull_xgb"]
# distribution_hndl = scst.distributions.weibull_min

# 2) Change the following to existing paths
output_dirs = [
    # "../scripts/TEST_100_Optuna_XGB_500/", # Oputuna with 500 trials - XGB configspace
    "../scripts/TEST_100_SMAC_NAS_500/", # SMAC with 500 trials - NAS configspace
    "../scripts/TEST_100_SMAC_NAS_100/", # SMAC with 100 trials - NAS configspace
]
save_dirs = [
    # "../results_optuna_XGB_500/",
    "../results_SMAC_NAS_500/",
    "../results_SMAC_NAS_100/",
    # "../results/"
]
conditions = [
    # "optuna_XGB_500",
    "SMAC_NAS_500",
    "SMAC_NAS_100",
    # "SMAC_NAS"
]
save_dir_combined = "../results_combined/"
os.makedirs(save_dir_combined, exist_ok=True)

# 3) Select whether to compute results for all scenarios or just a single scenario
SCENARIOS = []
SCENARIOS += ['clasp_factoring', ]
SCENARIOS += ['saps-CVVAR', ]
SCENARIOS += ['lpg-zeno', ]
SCENARIOS += ['yalsat_qcp', ]
SCENARIOS += ['spear_qcp', ]
SCENARIOS += ['spear_swgcp', ]
SCENARIOS += ['yalsat_swgcp', ]

load = True

In [ ]:
dir_res_dict = dict()

for output_dir in output_dirs:
    res_dict = dict()  
    if not os.path.isdir(output_dir):
        print("Directory does not exist: %s" % output_dir)
        continue
    print("Processing directory: %s" % output_dir)
    for fl in os.listdir(output_dir):
        if not ".100." in fl:
            # Select only one seed
            continue
        print(fl, output_dir)

        fl_path = os.path.join(output_dir, fl)
        
        if os.path.isdir(fl_path):
            continue
        
        if not "pkl" in fl_path:
            continue
        
        if "txt" in fl_path or "png" in fl_path:
            continue

        if not os.path.isfile(fl_path):
            fl_path = os.path.join(output_dir)

        with open(fl_path, "rb") as fh:
            pkl = pickle.load(fh)
            train_pred = np.array(pkl[0])
            val_pred = np.array(pkl[1])
            print(train_pred.shape, val_pred.shape)
            add_info = pkl[2]
            task = add_info["task"]
            model = add_info["model"]
            scenario = add_info["scenario"]
            fold = add_info["fold"]

            # Transform the xgboostlss parameter predictions
            if 'expon_xgb' in model:
                train_pred = 1/train_pred
                val_pred = 1/val_pred
            elif 'lognormal_xgb' in model:
                # Flip the parameter order
                train_pred = train_pred[:, ::-1]
                val_pred = val_pred[:, ::-1]
                # Convert log of location -> location
                train_pred[:, 1] = np.exp(train_pred[:, 1])
                val_pred[:, 1] = np.exp(val_pred[:, 1])
            elif 'weibull_xgb' in model:
                # Flip the parameter order
                train_pred = train_pred[:, ::-1]
                val_pred = val_pred[:, ::-1]
            elif 'invgauss_xgb' in model:
                ...
                # Flip the parameter order
                # train_pred = train_pred[:, ::-1]
                # val_pred = val_pred[:, ::-1]
                # # Pred_params: loc, shape -> scipy mu = shape = loc/scale, loc, scale = loc/shape
                # train_pred[:, 0] = train_pred[:, 1] / train_pred[:, 0]
                # val_pred[:, 0] = val_pred[:, 1] / val_pred[:, 0]

            if "nn" in add_info["model"] and add_info["task"] == "floc":
                print("%20s" % scenario, model, fold, add_info["loaded"])
        if scenario not in res_dict:
            res_dict[scenario] = dict()

        if model not in floc_exp and model not in mean_exp:
            continue

        if task not in res_dict[scenario]:
            res_dict[scenario][task] = dict()
            
        if model not in res_dict[scenario][task]:
            res_dict[scenario][task][model] = dict()

        res_dict[scenario][task][model][fold] = (train_pred, val_pred)
    
    # Store results for this directory
    dir_res_dict[output_dir] = res_dict

    if len(SCENARIOS) == 0:
        SCENARIOS = list(res_dict.keys())



In [ ]:
dir_nllh_dict = dict()
dir_mean_dict = dict()
dir_var_dict = dict()

for save_dir in save_dirs:
    nllh_dict = dict()
    mean_dict = dict()
    var_dict = dict()
    if load:
        with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
            nllh_dict = pickle.load(fh)
            
        with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
            mean_dict = pickle.load(fh)
            
        with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
            var_dict = pickle.load(fh)
    dir_nllh_dict[save_dir] = nllh_dict
    dir_mean_dict[save_dir] = mean_dict
    dir_var_dict[save_dir] = var_dict
        
print("LOADED")


In [ ]:
# Generate NLLH table

# nan means there are some folds missing
# inf means there was no results file found
tab_data = list()
header = ["sc"] + [sd[-7:] for sd in conditions]
for scen in sorted(nllh_dict.keys()):
    dummy_array = np.ones([2, 2]) * np.inf
    tab_data.append([scen] + [np.mean(nllh_dict[scen].get(floc_exp[0], dummy_array)[:, 0]) for nllh_dict in dir_nllh_dict.values()])
    tab_data.append(["", ] + [np.mean(nllh_dict[scen].get(floc_exp[0], dummy_array)[:, 1]) for nllh_dict in dir_nllh_dict.values()])

a = tabulate.tabulate(tab_data, headers=header, floatfmt="5.3f")
print(a)

with open("%s/nllh_%s.txt" % (save_dir_combined, distribution_hndl.name), "w") as fh:
    fh.write(a)